QA Data Generation using Iterative Calling.

Use case -
1. Generate dataset

In [12]:
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableConfig
from langgraph.graph import START, END, StateGraph
from typing import TypedDict, List

Load Document for Summarization 

In [13]:
def load_doc(file_path : str):
    document_loader = PyPDFLoader(file_path)
    documents = document_loader.load()
    return documents

In [14]:
##check
documents = load_doc("STM_8T.pdf") 
for doc in documents:
    print(doc.page_content)

April 2012 Doc ID 022351 Rev 2 1/22
PM0212
Programming manual
How to program the STM8TL5xxx
Flash program memory and data EEPROM
Introduction
This manual describes how to program Flash program memory and data EEPROM on 
STM8TL5xxx microcontrollers. It applies to STM8TL5xxx devices. It is intended to provide 
information to the programming tool manufacturers and to the customers who want to 
implement programming by themselves on their production line.
The in-circuit programming (ICP) method is used to update the content of Flash program 
memory and data EEPROM while the user software is not running. It uses the Single wire 
interface module (SWIM) to communicate between the programming tool and the device.
In contrast to the ICP method, in-application programming (IAP) can use any 
communication interface supported by the microcontroller (I/Os, SPI, USART, I
2C, USB, 
CAN...). IAP has been implemented for users who want their application software to update 
itself by re-programming the

The LLM

In [15]:
api_key = 'Eajkd7toYyYCEoU1LQiNFcPTvyK3ONep'
llm = ChatMistralAI(api_key=api_key, model_name= "mistral-large-latest")

The Nodes : generate, refine, END

In [16]:
class State(TypedDict):
    content : List[str]
    summary : str
    index : int

In [17]:
async def generate_qa(state : State):
    prompt = ChatPromptTemplate.from_template(""" With the given content, write possible question and answer pairs.
                                            {content}
                                            """)
    initial_summary_chain = prompt | llm 
    initial_summary = await initial_summary_chain.ainvoke(input={"content":state["content"][state["index"]]})
    return {"summary": initial_summary, "index":state["index"]+1}


The Router

In [18]:
def route(state:State):
    if state["index"] >= len(state["content"]):
        return END 
    else :
       return "generate_qa"

In [19]:
graph_builder = StateGraph(State)
graph_builder.add_node("generate_qa",generate_qa)
graph_builder.add_edge(START, "generate_qa")
graph_builder.add_conditional_edges("generate_qa", route)

In [20]:
graph = graph_builder.compile()
graph.get_graph()

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnablePassthrough(), metadata=None), 'generate_qa': Node(id='generate_qa', name='generate_qa', data=generate_qa(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}, edges=[Edge(source='__start__', target='generate_qa', data=None, conditional=False), Edge(source='generate_qa', target='__end__', data=None, conditional=False)])

INFERENCE

In [21]:
qa_summary=[]
qa_content=[]
async for step in graph.astream(
    {"content": [doc.page_content for doc in documents[3:]],
    "index":1},
    stream_mode="values",
):
    qa_summary.append(step.get("summary"))
    qa_content.append(step.get("content"))
    if summary := step.get("summary"): ## if the new summary is same as previous summary
        print(summary)

content='Sure, here are some possible question and answer pairs based on the given content:\n\n1. **Q:** What is the primary purpose of the Single Wire Interface Module (SWIM) in STM8 microcontrollers?\n   **A:** The primary purpose of the SWIM is to provide non-intrusive debug capability.\n\n2. **Q:** What additional functionalities does the SWIM protocol offer?\n   **A:** The SWIM protocol can also be used to download programs into RAM and execute them, write to registers or RAM, read any part of the memory space, and jump to any memory address.\n\n3. **Q:** How is the SWIM protocol accessed?\n   **A:** The SWIM protocol is accessed by providing a specific sequence on the SWIM pin either during the reset phase or when the device is running (if allowed by the application).\n\n4. **Q:** What is stored in the User Boot Code Area (UBC)?\n   **A:** The UBC contains the reset vector, interrupt vectors, and IAP routine to help the device recover from interrupted or erroneous IAP programming

In [22]:
for response in qa_summary[1:]:
    response.pretty_print()

================================== Ai Message ==================================

Sure, here are some possible question and answer pairs based on the given content:

1. **Q:** What is the primary purpose of the Single Wire Interface Module (SWIM) in STM8 microcontrollers?
   **A:** The primary purpose of the SWIM is to provide non-intrusive debug capability.

2. **Q:** What additional functionalities does the SWIM protocol offer?
   **A:** The SWIM protocol can also be used to download programs into RAM and execute them, write to registers or RAM, read any part of the memory space, and jump to any memory address.

3. **Q:** How is the SWIM protocol accessed?
   **A:** The SWIM protocol is accessed by providing a specific sequence on the SWIM pin either during the reset phase or when the device is running (if allowed by the application).

4. **Q:** What is stored in the User Boot Code Area (UBC)?
   **A:** The UBC contains the reset vector, interrupt vectors, and IAP routine to help the